Prompt Chaining with the langgraph. ( using the multiple prompt )

For this workflow . We will give a topic and generate the blog: 
For this: We will ask the llm for the outline of the blog with the first prompt . The we will generate the blog using the detailed outline  generated from the llm 

In [60]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv

In [61]:
load_dotenv()

True

In [62]:
model = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite")

In [63]:
# model = model.with_structured_output(BlogState)

In [64]:
class BlogState(TypedDict):

    title: str
    outline: str
    content: str
    rating: int
    

TypedDict only gives you a type hint. It does not validate that the LLM returns an integer between 1 and 5. So structed output with Pydantic

In [65]:


# from pydantic import BaseModel, Field


# class BlogState(BaseModel):
#     title: str
#     outline: str = ""
#     content: str = ""
#     rating: int | None = Field(
#         default=None,
#         ge=1,
#         le=5
#     )

In [66]:
def create_outline(state: BlogState) -> BlogState:

    # fetch title

    title = state['title']

    # call thellm gen outline 

    prompt = f'Generate a detailed but not long outline for a blog on the topic - {title}'
    outline = model.invoke(prompt).content[0]['text']
    # update state 
    state['outline'] = outline

    return state 

In [67]:
def create_blog(state: BlogState) -> BlogState:

    # fetch title

    title = state['title']

    # call the llm gen blog
    outline= state['outline']

    prompt = f'Generate a detailed blog on the topic - {title} using the following outline \n {outline}'

    content = model.invoke(prompt).content[0]['text']
   
    # update state 
    state['content'] = content

    return state 

In [68]:
def evaluate_blog(state: BlogState) -> BlogState:

    # fetch title

    title = state['title']

    # call thellm gen blog
    blog= state['content']

    prompt = f'Evaluate the  blog on the topic - {title} and content \n {blog} and give rating on 5 stars . ( Note the rating must be int between 1-5)'

    rating = model.invoke(prompt).content[0]['text']
   
    # update state 
    state['rating'] = rating

    return state 

In [69]:
graph = StateGraph(BlogState)

# nodes 

graph.add_node('create_outline',create_outline)
graph.add_node('create_blog',create_blog)
graph.add_node('evaluate_blog',evaluate_blog)


# edges 

graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline','create_blog')
graph.add_edge('create_blog','evaluate_blog')
graph.add_edge('evaluate_blog',END)

# compile 

workflow = graph.compile()

In [70]:
initial_state = {'title':'Nims Purja BroadPeak Incident'}
final_state = workflow.invoke(initial_state)

print(final_state)

{'title': 'Nims Purja BroadPeak Incident', 'outline': 'This outline provides a structured, balanced, and engaging approach to discussing the Nimsdai Purja/Broad Peak incident.\n\n---\n\n### **Blog Title Ideas:**\n*   *The Controversy at 8,000 Meters: Dissecting the Nims Purja Broad Peak Incident*\n*   *Climbing Ethics vs. Summits: What Really Happened on Broad Peak?*\n*   *Beyond the Peak: Analyzing the Allegations Against Nimsdai Purja*\n\n---\n\n### **I. Introduction**\n*   **The Context:** Briefly introduce Nims Purja’s status as a mountaineering icon (14 Peaks record holder).\n*   **The Incident:** Define the controversy—the 2021 Broad Peak expedition and the allegations of using supplemental oxygen while claiming to be "no-O2."\n*   **The Thesis:** The incident highlights a growing tension in high-altitude mountaineering between professional athleticism, commercial pressure, and the ethics of transparency.\n\n### **II. The Allegations Explained**\n*   **The Core Claim:** Details r

In [71]:
print(final_state['content'])

# Climbing Ethics vs. Summits: What Really Happened on Broad Peak?

In the rarified air of the "Death Zone"—the altitude above 8,000 meters where human life can no longer be sustained—there is very little room for error. There is even less room for ambiguity. Yet, in recent years, the world of high-altitude mountaineering has been thrust into a complex debate surrounding one of its most recognizable figures: Nimsdai “Nims” Purja.

The man who famously shattered the record for climbing all 14 of the world’s 8,000-meter peaks in just seven months is a global icon. However, his 2021 expedition to Broad Peak—the 12th highest mountain in the world—sparked a firestorm of controversy. At the heart of the debate is a simple but profound question: **Did he or did he not use supplemental oxygen?**

This incident serves as a lens through which we can examine the shifting ethics of modern mountaineering, where the line between professional athleticism, personal branding, and the purity of the spor

In [72]:
print(final_state['rating'])


This is a high-quality, well-structured piece of editorial journalism. It manages to remain objective while still highlighting the tension that makes the topic compelling.

Here is an evaluation of the blog based on key criteria:

### Strengths
*   **Tone and Balance:** You navigate a highly contentious topic without taking a "hit-piece" stance. You frame the argument around the *evolution of the sport* rather than merely attacking the individual, which gives the piece intellectual weight.
*   **The "Why" Factor:** Your section on the "Social Media Factor" and "The Absence of VAR" is the strongest part of the blog. It elevates the discussion from a "he-said-she-said" rumor mill to a thoughtful critique of how modern technology and commercialism are changing high-altitude mountaineering.
*   **Clarity and Flow:** The prose is professional and evocative ("the rarified air of the Death Zone," "the mountain as the ultimate judge"). The headings act as effective signposts, making the argume